## Topic: RecursiveCharacterTextSplitter

### Agenda

- 1. Introduction of RecursiveCharacterTextSplitter

- 2. Syntax and Separator

- 3. Practical Examples of RecursiveCharacterTextSplitter

- 4. Complete Summary


### 1. Introduction of RecursiveCharacterTextSplitter

- Definition:
    
    
    
- In LangChain:    
    - RecursiveCharacterTextSplitter is LangChain's most popular text splitter. It takes a long piece of text and recursively splits it using a hierarchy of separators — starting with the most natural breakpoints (paragraphs) and falling back to less ideal ones (characters) only when necessary.


    - RecursiveCharacterTextSplitter = hierarchical text splitting.


In [ ]:
""" 
Large Text
    │
    ▼
Try paragraph boundary
    │
    ├── Fits → Keep
    │
    └── Too large
          │
          ▼
     Try line boundary
          │
          ├── Fits → Keep
          │
          └── Too large
                │
                ▼
           Try word boundary
                │
                ├── Fits → Keep
                │
                └── Too large
                      │
                      ▼
                 Character level


"""

### 2. Syntax and Separator

In [ ]:
"""
# 1. Import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Create an object of RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    # 2.1: Custom Separator
    separators=[
        "\n\n",  # -> Paragraph
        "\n",    # -> Line
        " ",     # -> Word
        ""       # -> Character
    ],

    # 2.2: Chunk size parameter
    chunk_size=1000,

    # 2.3: Chunk overlap parameter
    chunk_overlap=200
)


"""

In [ ]:
"""        The Separator Hierarchy — The Heart of the Algorithm
        ==========================================================
- DEFAULT_SEPARATORS = ["\n\n", "\n", ". ", ", ", " ", ""]

- What Each Separator Means: 
            ┌──────────┬───────────────────────┬─────────────────────────────────────┐
            │ Priority │ Separator             │ What It Splits                      │
            ├──────────┼───────────────────────┼─────────────────────────────────────┤
            │ 1st      │ "\n\n"                │ PARAGRAPHS                          │
            │          │                       │ The BEST split — preserves full     │
            │          │                       │ paragraphs as coherent units.       │
            │          │                       │                                     │
            │          │                       │ "Paragraph about AI.\n\n"           │
            │          │                       │ "Paragraph about ML.\n\n"           │
            ├──────────┼───────────────────────┼─────────────────────────────────────┤
            │ 2nd      │ "\n"                  │ LINES                               │
            │          │                       │ Good for lists, code, poetry.       │
            │          │                       │                                     │
            │          │                       │ "Line 1\n"                          │
            │          │                       │ "Line 2\n"                          │
            ├──────────┼───────────────────────┼─────────────────────────────────────┤
            │ 3rd      │ ". "                  │ SENTENCES                           │
            │          │                       │ Splits at sentence boundaries.      │
            │          │                       │ Each chunk contains complete        │
            │          │                       │ sentences.                          │
            │          │                       │                                     │
            │          │                       │ "First sentence. "                  │
            │          │                       │ "Second sentence. "                 │
            ├──────────┼───────────────────────┼─────────────────────────────────────┤
            │ 4th      │ ", "                  │ CLAUSES                             │
            │          │                       │ Splits at commas — less ideal but   │
            │          │                       │ preserves clause structure.         │
            │          │                       │                                     │
            │          │                       │ "The cat, "                         │
            │          │                       │ "which was orange, "                │
            ├──────────┼───────────────────────┼─────────────────────────────────────┤
            │ 5th      │ " "                   │ WORDS                               │
            │          │                       │ Splits between words. Last resort   │
            │          │                       │ before character splitting.         │
            │          │                       │                                     │
            │          │                       │ "The quick "                        │
            │          │                       │ "brown fox "                        │
            ├──────────┼───────────────────────┼─────────────────────────────────────┤
            │ 6th      │ ""                    │ CHARACTERS                          │
            │          │                       │ NUCLEAR OPTION — splits between     │
            │          │                       │ individual characters. Only used    │
            │          │                       │ when a single word > chunk_size.    │
            │          │                       │                                     │
            │          │                       │ "S" "u" "p" "e" "r" "c" "a" "l"     │
            └──────────┴───────────────────────┴─────────────────────────────────────┘




"""

In [ ]:
"""   - The Recursion Explained:
    =================================
    
"Recursive" means the splitter CALLS ITSELF on oversized chunks.

Algorithm:
  1. Try to split with separator[0] ("\n\n")
  2. If any resulting chunk is still > chunk_size:
     → RECURSE: Try separator[1] ("\n") on that chunk
  3. If still too big:
     → RECURSE: Try separator[2] (". ") on that chunk
  4. Continue until chunk fits or you reach the last separator ("")
  5. Last separator ("") = split character by character (nuclear option)

This ensures EVERY chunk is ≤ chunk_size while preserving 
the most natural boundaries possible.


"""

In [ ]:
"""     - Chunk Size & Overlap Tuning

┌─────────────────────────────────────────────────────────────┐
│              CHUNK SIZE TUNING GUIDE                        │
│                                                             │
│  STEP 1: START WITH DEFAULTS                                │
│    chunk_size=1000, chunk_overlap=200                       │
│                                                             │
│  STEP 2: TEST WITH 10 SAMPLE QUERIES                        │
│    For each query, check:                                   │
│    a) Are the RIGHT chunks being retrieved?                 │
│    b) Do the chunks contain ENOUGH context?                 │
│    c) Do the chunks contain too much NOISE?                 │
│                                                             │
│  STEP 3: ADJUST BASED ON RESULTS                            │
│                                                             │
│    Problem: Retrieving irrelevant chunks                    │
│    → Chunks are too big (contain multiple topics)           │
│    → FIX: DECREASE chunk_size to 500-700                    │
│                                                             │
│    Problem: Answers are incomplete                          │
│    → Chunks are too small (missing context)                 │
│    → FIX: INCREASE chunk_size to 1200-1500                  │
│                                                             │
│    Problem: Sentences cut in half                           │
│    → Not enough overlap                                     │
│    → FIX: INCREASE chunk_overlap to 20-25% of chunk_size    │
│                                                             │
│    Problem: Too many chunks (slow retrieval)                │
│    → Chunks are too small                                   │
│    → FIX: INCREASE chunk_size, DECREASE overlap             │
│                                                             │
│    Problem: High embedding costs                            │
│    → Too many chunks                                        │
│    → FIX: INCREASE chunk_size to reduce total chunks        │
│                                                             │
│  STEP 4: VALIDATE WITH REAL USERS                           │
│    → Track answer quality, latency, and cost                │
│    → Iterate based on feedback                              │
└─────────────────────────────────────────────────────────────┘


"""

In [ ]:
"""         - Visual Comparison — Different Chunk Sizes

Original Text (1000 chars):
    "Paragraph 1 about AI in healthcare. AI is used for diagnosis 
    and treatment planning. Paragraph 2 about AI in finance. ML 
    models detect fraud and predict markets. Paragraph 3 about AI 
    in education. Personalized learning adapts to student needs."

    
    chunk_size=200, overlap=30:
    ┌──────────────────┐
    │ Chunk 1: Para 1  │ 200 chars
    │   ...treatment.  │
    └────┬─────────────┘
         │ 30 chars overlap
    ┌────▼─────────────┐
    │ Chunk 2: Para 2  │ 200 chars
    │   ...markets.    │
    └────┬─────────────┘
         │ 30 chars overlap
    ┌────▼─────────────┐
    │ Chunk 3: Para 3  │ 200 chars
    │   ...needs.      │
    └──────────────────┘
    Result: 5 chunks, very precise retrieval, less context per chunk

    chunk_size=500, overlap=100:
    ┌──────────────────────────────┐
    │ Chunk 1: Para 1 + Para 2     │ 500 chars
    │   ...predict markets.        │
    └────┬─────────────────────────┘
         │ 100 chars overlap
    ┌────▼─────────────────────────┐
    │ Chunk 2: Para 2 + Para 3     │ 500 chars
    │   ...student needs.          │
    └──────────────────────────────┘
    Result: 2 chunks, good context, moderate precision

    chunk_size=1000, overlap=200:
    ┌────────────────────────────────────────────┐
    │ Chunk 1: All paragraphs                    │ 1000 chars
    │   ...student needs.                        │
    └────────────────────────────────────────────┘
    Result: 1 chunk, maximum context, low precision


"""

### 3. Practical Examples of RecursiveCharacterTextSplitter

In [7]:
# Example 1: RecursiveCharacterTextSplitter
# Purpose: Basic Operation of RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
Space exploration has led to incredible scientific discoveries. From landing on the Moon to exploring Mars, humanity continues to push the boundaries of what's possible beyond our planet.

These missions have not only expanded our knowledge of the universe but have also contributed to advancements in technology here on Earth. Satellite communications, GPS, and even certain medical imaging techniques trace their roots back to innovations driven by space programs.
"""

# Initialize RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    # separator = "",
    chunk_size = 50,
    chunk_overlap = 5
)

# Perform the split
chunks = splitter.split_text(text)

print(f"Length of chunk: {len(chunks)}")
print(f"2nd chunk data: {chunks[1]}")
print(f"3rd chunk data: {chunks[2]}")

Length of chunk: 12
2nd chunk data: scientific discoveries. From landing on the Moon
3rd chunk data: Moon to exploring Mars, humanity continues to


### ###  Example — Multi-Language Documents 

In [7]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a token-counting function
encoding = tiktoken.get_encoding("cl100k_base")

def token_count(text: str) -> int:
    return len(encoding.encode(text))


# Create a token-aware splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=152,          # 500 TOKENS, not characters
    chunk_overlap=20,
    length_function=token_count,  # ← Use token counting!
    separators=["\n\n", "\n", ". ", "。", "، ", "! ", "? ", " ", ""]
    # ↑ Added "。" (Japanese period) and "،" (Arabic comma)
)

multilingual_text = """
Artificial intelligence is transforming industries worldwide. 
Machine learning models can now process text, images, and audio 
with remarkable accuracy.

人工知能は世界中の産業を変革しています。機械学習モデルは、テキスト、画像、 
音声を驚異的な精度で処理できるようになりました。深層学習の進歩により、 
自然言語処理とコンピュータビジョンの分野で大きな飛躍が見られました。

الذكاء الاصطناعي يحول الصناعات في جميع أنحاء العالم. يمكن لنماذج 
التعلم الآلي الآن معالجة النصوص والصور والصوت بدقة ملحوظة.
"""

chunks = splitter.split_text(multilingual_text)


print(f"Characters: {len(chunks)}")   # 42 characters
print(f"Tokens: {token_count(multilingual_text)}")  # ~18 tokens (multilingual uses more!)

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Characters: {len(chunk)} | Tokens: {token_count(chunk)}")
    print(chunk[:152])

Characters: 3
Tokens: 245

--- Chunk 1 ---
Characters: 153 | Tokens: 26
Artificial intelligence is transforming industries worldwide. 
Machine learning models can now process text, images, and audio 
with remarkable accuracy

--- Chunk 2 ---
Characters: 110 | Tokens: 128
人工知能は世界中の産業を変革しています。機械学習モデルは、テキスト、画像、 
音声を驚異的な精度で処理できるようになりました。深層学習の進歩により、 
自然言語処理とコンピュータビジョンの分野で大きな飛躍が見られました。

--- Chunk 3 ---
Characters: 124 | Tokens: 90
الذكاء الاصطناعي يحول الصناعات في جميع أنحاء العالم. يمكن لنماذج 
التعلم الآلي الآن معالجة النصوص والصور والصوت بدقة ملحوظة.


### 4. Complete Summary RecursiveCharacterTextSplitter

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│              RecursiveCharacterTextSplitter                      │
│                                                                  │
│  WHAT:  Recursively splits text using a hierarchy of separators  │
│  WHY:   Creates semantically coherent chunks within size limits  │
│  WHERE: After Loaders, before Embeddings in RAG pipeline         │
│                                                                  │
│  ALGORITHM:                                                      │
│    1. Try "\n\n" (paragraphs) → If chunk fits, done              │
│    2. If too big, try "\n" (lines) → If fits, done               │
│    3. If too big, try ". " (sentences) → If fits, done           │
│    4. If too big, try ", " (clauses) → ...                       │
│    5. If too big, try " " (words) → ...                          │
│    6. Nuclear option: "" (characters)                            │
│                                                                  │
│  KEY PARAMS:                                                     │
│    chunk_size    → Max chars per chunk (recommended: 800-1500)   │
│    chunk_overlap → Shared chars (recommended: 10-20% of size)    │
│    separators    → Split hierarchy (customize per content type)  │
│    length_function → How to measure size (len or token count)    │
│                                                                  │
│  METHODS:                                                        │
│    .split_text(str)         → Returns List[str]                  │
│    .split_documents(docs)   → Returns List[Document]             │
│    .from_language(Language) → Code-aware splitter                │
│    .get_separators_for_language() → Get default seps             │
│                                                                  │
│  RECOMMENDED DEFAULTS:                                           │
│    RecursiveCharacterTextSplitter(                               │
│        chunk_size=1000,                                          │
│        chunk_overlap=200,                                        │
│        separators=["\n\n", "\n", ". ", ", ", " ", ""]            │ 
│    )                                                             │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "Start with 1000/200. Print 10 random chunks. If they look      │
│   good, you're done. If not, tune size and separators."          │
└──────────────────────────────────────────────────────────────────┘

"""

In [ ]:
"""     - Performance Benchmarks
┌──────────────────────┬───────────────┬───────────────┬──────────────┐
│ Document Size        │ Pages         │ Split Time    │ Chunks       │
├──────────────────────┼───────────────┼───────────────┼──────────────┤
│ Short article        │ 2 pages       │ < 1ms         │ ~8           │
│ Research paper       │ 15 pages      │ ~5ms          │ ~60          │
│ Company report       │ 50 pages      │ ~15ms         │ ~200         │
│ Technical manual     │ 200 pages     │ ~50ms         │ ~800         │
│ Book                 │ 500 pages     │ ~120ms        │ ~2000        │
│ Large codebase       │ 1000 files    │ ~500ms        │ ~5000        │
└──────────────────────┴───────────────┴───────────────┴──────────────┘

Note: Splitting is EXTREMELY fast (pure string operations).
The bottleneck is always EMBEDDING, not splitting.


"""